# Colab 29 — Unified comparison: method × metric × alphabet

**One harness, one shared eval, apples-to-apples.** Every method is scored on the *same pool and the
same pairs* (per feed) under the *same three metrics*, so the deck can show a method × metric table
instead of stitching numbers across notebooks.

- **Methods (rows):** trigram (count of shared 3-grams), Dice, length-only floor, ESM2, **SNN (ours)**;
  ProtTrans/ProtT5 = stubbed WIP row (flip `ENABLE_PROTTRANS`).
- **Alphabets (feeds):** AA / SS / 3Di.
- **Metrics:** **Spearman ρ(sim, normLev)** — primary, threshold-free, on a *stratified full-range*
  pair set (equal-per-normLev-bin, **stratified per feed by that feed's own normLev**);
  **AUROC** (is-high ≥ 0.70 vs low/mid); **MAP@10 / hit@10** (full-pool exhaustive de-hubbed oracle,
  bars 0.70 and 0.90, with the length floor).

Ground truth is **exact normLev on our own strings** only (Fenoy's 0.66 is vs BLAST — context, not a
direct comparison). Add a method = add one entry to `METHODS`; all three metrics + figures run over it.

*Locked design + result snapshot in `PRESENTATION_PLAN_v12.md`. Reuses the colab24f/colab28 harness verbatim.*

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
CACHE = '/content/bench_cache'; os.makedirs(CACHE, exist_ok=True)
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy transformers sentencepiece matplotlib --quiet

In [ ]:
import time, json, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.sparse as sp
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED); rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# ---- toggles ----
ENABLE_PROTTRANS = False   # flip True to add the ProtT5 row (needs GPU + sentencepiece; ~2.8B params)
N_STRAT_PER_BIN  = 400     # target pairs per normLev decile bin, per feed (capped by availability)
N_STRAT_CAND     = 300_000 # random candidate pairs sampled per feed before binning
N_BOOT           = 1000    # bootstrap resamples for MAP@10 CI

## 2. Constants + helpers + SNN architecture (colab24e/28 recipe, verbatim)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN = 50, 200; N_TRAIN, EPOCHS, BS, K = 30000, 30, 128, 16
BAND_LOW_AA, BAND_LOW_SS, BAND_HIGH, BAND_STRICT = 0.30, 0.56, 0.70, 0.90
BAND_MID = 0.30   # lower edge of the hard-negative window [0.30, 0.70) for full-pool AUROC
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, abc):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def rand_seq(abc): L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))
def bin_idx(x, band_low): return 0 if x < band_low else (1 if x < BAND_HIGH else 2)

In [ ]:
class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)
class Clf(nn.Module):
    def __init__(s):
        super().__init__(); s.encoder = Enc()
        s.head = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 3))
    def forward(s, a, b): return s.head(torch.abs(s.encoder(a) - s.encoder(b)))
class DS(Dataset):
    def __init__(s, pp, bl): s.p = pp; s.bl = bl
    def __len__(s): return len(s.p)
    def __getitem__(s, i): a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), bin_idx(l, s.bl)

def train_snn(alphabet, band_low, label):
    t0 = time.perf_counter(); pairs = []
    while len(pairs) < N_TRAIN:
        sd = rand_seq(alphabet); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, alphabet)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    datagen_s = time.perf_counter() - t0
    dl = DataLoader(DS(pairs, band_low), batch_size=BS, shuffle=True)
    model = Clf().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
    model.train(); t0 = time.perf_counter()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'  [{label}] epoch {ep}/{EPOCHS} CE {tot/nb:.4f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval(); return model, datagen_s, time.perf_counter() - t0
snn_params = sum(p.numel() for p in Enc().parameters())
print(f'SNN encoder params = {snn_params:,}')

## 3. Per-representation pools (colab24f/28 protocol, verbatim)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])            if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])            if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_IDS = {f: list(LOOK[f]) for f in LOOK}
POOL_SEQ = {f: [LOOK[f][d] for d in POOL_IDS[f]] for f in LOOK}
POOL_LEN = {f: np.array([len(s) for s in POOL_SEQ[f]]) for f in LOOK}
FEEDS = ['AA', 'SS', '3Di']
for f in FEEDS: print(f'  {f:<4} pool = {len(POOL_IDS[f]):>6}')

## 4. Train the two SNN encoders (frozen afterwards); 3Di reuses the AA encoder

In [ ]:
model_aa, _, _ = train_snn(AA_ALPHABET, BAND_LOW_AA, 'AA')
model_ss, _, _ = train_snn(SS_ALPHABET, BAND_LOW_SS, 'SS')
SNN_BY_FEED = {'AA': model_aa, 'SS': model_ss, '3Di': model_aa}

## 5. Exhaustive Levenshtein oracle (de-hubbed true-neighbour sets, bars 0.70 & 0.90)

Same all-pairs exhaustive protocol as colab24f: full pool × pool exact Levenshtein, blocked. Returns
true-neighbour sets at each bar (self excluded) and, as a by-product, every high-sim positive pair
(used to guarantee the top normLev bins are populated for the Spearman set, and as AUROC positives).

In [ ]:
def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = POOL_LEN[feed]; N = len(seqs)
    T_high, T_strict = {}, {}
    pos_pairs = []   # (i, j, normLev) for j>i with normLev >= BAND_HIGH  (Spearman top-bin + AUROC pos)
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            st = np.where(row >= BAND_STRICT)[0]
            if st.size: T_strict[i] = st.astype(np.int32)
            for j in hi:
                if j > i: pos_pairs.append((i, int(j), float(row[j])))
    med_h = int(np.median([len(v) for v in T_high.values()])) if T_high else 0
    print(f'  oracle[{feed}]: queries@0.70={len(T_high)} (med|T|={med_h}), '
          f'queries@0.90={len(T_strict)}, high-sim pos pairs={len(pos_pairs)}')
    return dict(T_high=T_high, T_strict=T_strict, pos_pairs=pos_pairs)
print('Building exact-Levenshtein oracle (exhaustive, de-hubbed)...')
ORACLE = {f: build_oracle(f) for f in FEEDS}

## 6. Shared stratified pair set — per feed, by that feed's OWN normLev

For each feed: sample many random pool pairs, compute normLev, bin into deciles [0,.1)…[.9,1], and
draw up to `N_STRAT_PER_BIN` per populated bin. Guaranteed to include the exhaustive high-sim pairs so
the top bins are covered even for AA (where natural high-sim pairs are extremely rare). **This exact
pair set is reused by every method** so the Spearman/AUROC comparison is fair. AA's top bins will be
sparse by construction — that scarcity is the data reality, reported, not hidden.

In [ ]:
def build_strat_pairs(feed):
    N = len(POOL_SEQ[feed]); seqs = POOL_SEQ[feed]
    # 1) random candidate pairs
    a = rng.integers(0, N, size=N_STRAT_CAND); b = rng.integers(0, N, size=N_STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    # 2) add exhaustive high-sim positives (guarantees top-bin coverage) — build once, concat once
    pp = ORACLE[feed]['pos_pairs']
    if pp:
        parr = np.array(pp, dtype=float)   # (P,3): i, j, normLev
        a = np.concatenate([a, parr[:, 0].astype(np.int64)])
        b = np.concatenate([b, parr[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, parr[:, 2]])
    # 3) stratify into deciles
    edges = np.linspace(0.0, 1.0, 11)
    bins = np.clip(np.digitize(nl, edges) - 1, 0, 9)
    ai, aj, av, binlist = [], [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size == 0: continue
        take = rng.permutation(idx)[:N_STRAT_PER_BIN]
        ai.append(a[take]); aj.append(b[take]); av.append(nl[take]); binlist.append(np.full(take.size, bb))
    ai = np.concatenate(ai); aj = np.concatenate(aj); av = np.concatenate(av); binlist = np.concatenate(binlist)
    return dict(i=ai.astype(np.int64), j=aj.astype(np.int64), nl=av, bins=binlist)

STRAT = {f: build_strat_pairs(f) for f in FEEDS}
print('Stratified pair set per feed (count per normLev decile):')
hdr = 'feed  ' + ''.join(f'[{k/10:.1f})'.rjust(7) for k in range(10)) + '   total'
print(hdr); print('-'*len(hdr))
for f in FEEDS:
    counts = [int((STRAT[f]['bins'] == b).sum()) for b in range(10)]
    print(f'{f:<5} ' + ''.join(str(c).rjust(7) for c in counts) + f'   {sum(counts):>6}')

## 7. Method registry — add a method = add one entry

Each method exposes two vectorized calls over a feed's pool:
- `pair(feed, i, j)` → similarity for pair arrays (used by Spearman + AUROC),
- `query(feed, qi)` → similarity of pool item `qi` vs the whole pool (used by retrieval; self set to −∞).

Embedding methods (SNN, ESM2, ProtTrans) share one implementation over an L2-normalised pool matrix
(cosine = dot). Spearman/AUROC/retrieval are all rank-based, so cosine is metric-equivalent to the
SNN's `1−L2/2` convention. String methods (trigram/Dice) use a sparse binary k-mer matrix. Length is
the trivial floor.

In [ ]:
# ---------- embedding backends ----------
@torch.no_grad()
def snn_pool(feed):
    ids = POOL_IDS[feed]; look = LOOK[feed]; model = SNN_BY_FEED[feed]
    out = np.zeros((len(ids), 128), np.float32); model.eval()
    for i in range(0, len(ids), 256):
        x = torch.stack([encode_pad(look[d]) for d in ids[i:i+256]]).to(device)
        out[i:i+x.shape[0]] = model.encoder(x).cpu().numpy()
    return out

ESM2_MODEL = 'facebook/esm2_t12_35M_UR50D'
_esm = {}
def _load_esm():
    if 'mdl' in _esm: return
    from transformers import AutoTokenizer, AutoModel
    _esm['tok'] = AutoTokenizer.from_pretrained(ESM2_MODEL)
    _esm['mdl'] = AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
@torch.no_grad()
def esm_pool(feed):
    _load_esm(); tok, mdl = _esm['tok'], _esm['mdl']
    cf = f'{CACHE}/esm2_29_{feed}.npy'; sf = f'{CACHE}/esm2_29_{feed}.meta.json'
    ids = POOL_IDS[feed]
    if os.path.exists(cf) and os.path.exists(sf):
        meta = json.load(open(sf))
        if meta.get('model') == ESM2_MODEL and meta.get('ids') == ids: return np.load(cf)
    seqs = POOL_SEQ[feed]; order = np.argsort([len(s) for s in seqs]); out = [None]*len(seqs)
    for i in range(0, len(order), 32):
        idx = order[i:i+32]; batch = [seqs[j] for j in idx]
        enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=True).to(device)
        h = mdl(**enc).last_hidden_state; mask = enc['attention_mask'].clone(); mask[:, 0] = 0
        for r, l in enumerate(enc['attention_mask'].sum(1)): mask[r, l-1] = 0
        m = mask.unsqueeze(-1).float(); e = (h*m).sum(1)/m.sum(1).clamp(min=1)
        e = F.normalize(e, dim=1).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    e = np.stack(out).astype(np.float32); np.save(cf, e)
    json.dump({'model': ESM2_MODEL, 'ids': ids}, open(sf, 'w')); return e

PROTTRANS_MODEL = 'Rostlab/prot_t5_xl_half_uniref50-enc'
_pt = {}
def prot_pool(feed):
    # WIP row. Fed the same way as ESM2 (all three feeds tokenise as AA-letter strings).
    if 'mdl' not in _pt:
        from transformers import T5Tokenizer, T5EncoderModel
        _pt['tok'] = T5Tokenizer.from_pretrained(PROTTRANS_MODEL, do_lower_case=False)
        _pt['mdl'] = T5EncoderModel.from_pretrained(PROTTRANS_MODEL).to(device).eval()
    tok, mdl = _pt['tok'], _pt['mdl']; seqs = POOL_SEQ[feed]
    out = []
    with torch.no_grad():
        for i in range(0, len(seqs), 16):
            batch = [' '.join(list(s)) for s in seqs[i:i+16]]
            enc = tok(batch, return_tensors='pt', padding=True).to(device)
            h = mdl(**enc).last_hidden_state; m = enc['attention_mask'].unsqueeze(-1).float()
            e = (h*m).sum(1)/m.sum(1).clamp(min=1)
            out.append(F.normalize(e, dim=1).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

# ---------- sparse k-mer backend (trigram / Dice) ----------
def build_kmer(feed, k=3):
    seqs = POOL_SEQ[feed]; vocab = {}; rows, cols = [], []
    for r, s in enumerate(seqs):
        seen = set(s[t:t+k] for t in range(len(s)-k+1))
        for g in seen:
            c = vocab.setdefault(g, len(vocab)); rows.append(r); cols.append(c)
    B = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)),
                      shape=(len(seqs), max(1, len(vocab))))
    sizes = np.asarray(B.sum(1)).ravel()
    return B, sizes

# ---------- caches ----------
EMB = {}      # (backend, feed) -> pool matrix
KMER = {}     # feed -> (B, sizes)
def get_emb(backend, feed):
    key = (backend, feed)
    if key not in EMB:
        EMB[key] = {'snn': snn_pool, 'esm': esm_pool, 'prot': prot_pool}[backend](feed)
    return EMB[key]
def get_kmer(feed):
    if feed not in KMER: KMER[feed] = build_kmer(feed)
    return KMER[feed]

In [ ]:
# ---------- unified similarity interface ----------
# Each method exposes: pair(feed,i,j) [Spearman], query(feed,qi) [retrieval, self=-inf],
# block(feed,r0,r1) [full-pool AUROC: rows r0:r1 vs all cols, shape (r1-r0, N)].
def _emb_pair(be, feed, i, j):
    E = get_emb(be, feed); return np.sum(E[i]*E[j], axis=1)
def _emb_query(be, feed, qi):
    E = get_emb(be, feed); s = E @ E[qi]; s[qi] = -np.inf; return s
def _emb_block(be, feed, r0, r1):
    E = get_emb(be, feed); return E[r0:r1] @ E.T

def _kmer_pair(feed, i, j, mode):
    B, sz = get_kmer(feed)
    inter = np.asarray(B[i].multiply(B[j]).sum(1)).ravel().astype(float); sa, sb = sz[i], sz[j]
    if mode == 'count': return inter                                    # shared 3-gram COUNT (literal)
    if mode == 'dice':  return 2*inter/np.maximum(sa+sb, 1e-9)          # Dice (length-fair)
    return inter/np.maximum(sa+sb-inter, 1e-9)                          # Jaccard (optional)
def _kmer_query(feed, qi, mode):
    B, sz = get_kmer(feed)
    inter = np.asarray(B.dot(B[qi].T).todense()).ravel().astype(float); sq = sz[qi]
    if mode == 'count': s = inter
    elif mode == 'dice': s = 2*inter/np.maximum(sz+sq, 1e-9)
    else: s = inter/np.maximum(sz+sq-inter, 1e-9)
    s[qi] = -np.inf; return s
def _kmer_block(feed, r0, r1, mode):
    B, sz = get_kmer(feed)
    C = np.asarray(B[r0:r1].dot(B.T).todense()).astype(float); a = sz[r0:r1][:, None]; b = sz[None, :]
    if mode == 'count': return C
    if mode == 'dice':  return 2*C/np.maximum(a+b, 1e-9)
    return C/np.maximum(a+b-C, 1e-9)

def _len_pair(feed, i, j):
    L = POOL_LEN[feed]; return 1.0 - np.abs(L[i]-L[j])/np.maximum(np.maximum(L[i], L[j]), 1)
def _len_query(feed, qi):
    L = POOL_LEN[feed]; s = 1.0 - np.abs(L-L[qi])/np.maximum(np.maximum(L, L[qi]), 1)
    s = s.astype(float); s[qi] = -np.inf; return s
def _len_block(feed, r0, r1):
    L = POOL_LEN[feed]; a = L[r0:r1][:, None]; b = L[None, :]
    return 1.0 - np.abs(a-b)/np.maximum(np.maximum(a, b), 1)

# name -> dict(pair, query, block, status). Both string baselines are TRIGRAM (3-gram) measures, as
# the professor asked: 'trigram' = COUNT of shared 3-grams |A∩B| (the literal ask; note it is
# length-biased — longer sequences share more); 'Dice' = 2|A∩B|/(|A|+|B|), the length-fair trigram
# comparator that keeps the SNN honest. (A Jaccard variant is one flag away: mode='jaccard'.)
METHODS = {
    'trigram':   dict(pair=lambda f,i,j:_kmer_pair(f,i,j,'count'), query=lambda f,q:_kmer_query(f,q,'count'), block=lambda f,r0,r1:_kmer_block(f,r0,r1,'count'), status='live'),
    'Dice':      dict(pair=lambda f,i,j:_kmer_pair(f,i,j,'dice'),  query=lambda f,q:_kmer_query(f,q,'dice'),  block=lambda f,r0,r1:_kmer_block(f,r0,r1,'dice'),  status='live'),
    'length':    dict(pair=_len_pair,                              query=_len_query,                          block=_len_block,                                  status='live'),
    'ESM2':      dict(pair=lambda f,i,j:_emb_pair('esm',f,i,j),    query=lambda f,q:_emb_query('esm',f,q),    block=lambda f,r0,r1:_emb_block('esm',f,r0,r1),    status='live'),
    'SNN':       dict(pair=lambda f,i,j:_emb_pair('snn',f,i,j),    query=lambda f,q:_emb_query('snn',f,q),    block=lambda f,r0,r1:_emb_block('snn',f,r0,r1),    status='live'),
    'ProtTrans': dict(pair=lambda f,i,j:_emb_pair('prot',f,i,j),   query=lambda f,q:_emb_query('prot',f,q),   block=lambda f,r0,r1:_emb_block('prot',f,r0,r1),   status='wip'),
}
ACTIVE = [m for m, d in METHODS.items() if d['status'] == 'live' or (m == 'ProtTrans' and ENABLE_PROTTRANS)]
print('Active methods:', ACTIVE)
print('WIP (skipped, shown as WIP in tables):', [m for m in METHODS if m not in ACTIVE])

## 8. Metric 1 — Spearman ρ(sim, normLev)  [PRIMARY, centerpiece]

Threshold-free, whole-range, on the shared stratified pair set (per feed). Rank-based, so cosine vs
`1−L2/2` doesn't matter. This is the table that answers all three questions at once.

In [ ]:
spear_rows = []
for feed in FEEDS:
    P = STRAT[feed]; i, j, y = P['i'], P['j'], P['nl']
    for m in ACTIVE:
        s = METHODS[m]['pair'](feed, i, j)
        rho = spearmanr(s, y).correlation
        spear_rows.append(dict(method=m, feed=feed, rho=rho, n_pairs=len(y)))
spear = pd.DataFrame(spear_rows)
SPEAR_TAB = spear.pivot(index='method', columns='feed', values='rho').reindex(
    [m for m in METHODS if m in ACTIVE])[FEEDS]
print('='*60); print('SPEARMAN rho(sim, normLev)  — stratified full-range, per feed'); print('='*60)
print(SPEAR_TAB.round(3).to_string())
spear.to_csv('colab29_spearman.csv', index=False)

## 9. Metric 2 — AUROC (full-pool exhaustive, de-hubbed — matches colab24f, NOT the Spearman sample)

**Locked design:** AUROC is the full-pool sanity check, so it streams the *entire* pairwise grid (not
the stratified Spearman pairs). For each feed we block over rows, label every unordered pair by exact
normLev, and accumulate each method's predicted-similarity histograms. **Headline = high (≥0.70) vs
random negative**; **hard-negative [0.30, 0.70) is annotated** (the harder, more honest contrast). This
is the slide-14 "is the task easy?" check — watch trigram-count/Dice collapse toward chance on SS
(3³=27) while holding up on AA/3Di.

In [ ]:
NBINS = 4000; LO, HI = -0.10, 1.10
def _binidx(p):
    return np.clip(np.floor((p - LO)/(HI - LO)*NBINS).astype(np.int64), 0, NBINS-1)
def _auroc_hist(h_pos, h_neg):
    P, Nn = h_pos.sum(), h_neg.sum()
    if P == 0 or Nn == 0: return float('nan')
    cum_below = np.cumsum(h_neg) - h_neg
    return float((h_pos*(cum_below + 0.5*h_neg)).sum()/(P*Nn))

MAX_KMER_BIN = MAX_LEN   # raw shared-3gram count is integer <= MAX_LEN-2, so integer bins are EXACT
def _score_bins(method, s):
    # 'trigram' is an unbounded integer count -> bin on its own integer scale, else it clips into the
    # last float-bin and its AUROC is meaningless. Everything else lives in ~[0,1]/cosine -> float grid.
    if method == 'trigram': return np.clip(s.astype(np.int64), 0, MAX_KMER_BIN)
    return _binidx(s)

def auroc_fullpool(feed, block=512):
    seqs = POOL_SEQ[feed]; lens = POOL_LEN[feed]; N = len(seqs)
    H = {m: {'pos': np.zeros(NBINS), 'rand': np.zeros(NBINS), 'hard': np.zeros(NBINS)} for m in ACTIVE}
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        tsim = 1.0 - Dm/den
        S = {m: METHODS[m]['block'](feed, r0, r1) for m in ACTIVE}
        for a in range(r1 - r0):
            i = r0 + a
            if i + 1 >= N: continue
            tj = tsim[a, i+1:]                       # each unordered pair counted once (j > i)
            pos = tj >= BAND_HIGH; neg = ~pos; hard = (tj >= BAND_MID) & (tj < BAND_HIGH)
            for m in ACTIVE:
                sj = S[m][a, i+1:]
                if neg.any():  H[m]['rand'] += np.bincount(_score_bins(m, sj[neg]),  minlength=NBINS)
                if pos.any():  H[m]['pos']  += np.bincount(_score_bins(m, sj[pos]),  minlength=NBINS)
                if hard.any(): H[m]['hard'] += np.bincount(_score_bins(m, sj[hard]), minlength=NBINS)
    npos = int(H[ACTIVE[0]]['pos'].sum())
    return [dict(method=m, feed=feed, n_pos=npos,
                 auroc_random=_auroc_hist(H[m]['pos'], H[m]['rand']),
                 auroc_hard=_auroc_hist(H[m]['pos'], H[m]['hard'])) for m in ACTIVE]

print('Streaming full-pool AUROC (exhaustive)...')
auroc = pd.DataFrame([r for feed in FEEDS for r in auroc_fullpool(feed)])
AUROC_TAB = auroc.pivot(index='method', columns='feed', values='auroc_random').reindex(
    [m for m in METHODS if m in ACTIVE])[FEEDS]
print('='*64); print('AUROC (full-pool) — high (>=0.70) vs RANDOM negative [headline]'); print('='*64)
print(AUROC_TAB.round(3).to_string())
HARD_TAB = auroc.pivot(index='method', columns='feed', values='auroc_hard').reindex(
    [m for m in METHODS if m in ACTIVE])[FEEDS]
print('\nAUROC vs HARD negative [0.30,0.70) [annotation]:'); print(HARD_TAB.round(3).to_string())
print('\nn_pos (high pairs) per feed:', {f: int(auroc[auroc.feed==f]['n_pos'].iloc[0]) for f in FEEDS})
auroc.to_csv('colab29_auroc.csv', index=False)

## 10. Metric 3 — retrieval MAP@10 / hit@10 (full-pool, de-hubbed, bars 0.70 & 0.90)

Per query, rank the whole pool by each method's `query()` score; score AP@10 and hit@10 against the
exhaustive Levenshtein neighbour set. Bootstrap CI over queries. `length` is included as the floor.

In [ ]:
def _ap(order, ts, k=10):
    nt = len(ts)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for r, o in enumerate(order[:k], 1):
        if o in ts: hits += 1; ap += hits/r
    return ap/min(nt, k)

def retrieval(feed, method, bar):
    T = ORACLE[feed]['T_high'] if bar == 0.70 else ORACLE[feed]['T_strict']
    q_ids = list(T.keys())
    if not q_ids: return dict(bar=bar, feed=feed, method=method, n_q=0, med_T=0,
                              map=np.nan, map_lo=np.nan, map_hi=np.nan, hit=np.nan)
    qfn = METHODS[method]['query']; aps, hits = [], []
    for qi in q_ids:
        ts = set(T[qi].tolist()); order = np.argsort(-qfn(feed, qi))
        aps.append(_ap(order, ts, 10)); hits.append(1.0 if len(set(order[:10].tolist()) & ts) else 0.0)
    aps = np.array(aps)
    boot = np.array([np.nanmean(aps[rng.integers(0, len(aps), len(aps))]) for _ in range(N_BOOT)])
    return dict(bar=bar, feed=feed, method=method, n_q=len(q_ids),
                med_T=int(np.median([len(v) for v in T.values()])),
                map=float(np.nanmean(aps)), map_lo=float(np.percentile(boot, 2.5)),
                map_hi=float(np.percentile(boot, 97.5)), hit=float(np.mean(hits)))

retr_rows = []
for bar in (0.70, 0.90):
    for feed in FEEDS:
        for m in ACTIVE:
            retr_rows.append(retrieval(feed, m, bar))
retr = pd.DataFrame(retr_rows)
order = [m for m in METHODS if m in ACTIVE]
for bar in (0.70, 0.90):
    sub = retr[retr.bar == bar]
    print('='*72); print(f'RETRIEVAL @ bar {bar:.2f}'); print('='*72)
    print('MAP@10 with 95% bootstrap CI  (use for SS/3Di — neighbourhood retrieval; CI = receipt for "non-overlapping" claims):')
    ci = sub.assign(cell=sub.apply(lambda r: f"{r['map']:.3f} [{r['map_lo']:.3f},{r['map_hi']:.3f}]", axis=1))
    print(ci.pivot(index='method', columns='feed', values='cell').reindex(order)[FEEDS].to_string())
    print('\nhit@10  (use for AA — pair-like, median|T|=1):')
    print(sub.pivot(index='method', columns='feed', values='hit').reindex(order)[FEEDS].round(3).to_string())
    print()
print('METRIC SPLIT for slides: report AA as hit@10 (one true partner per query); '
      'report SS/3Di as MAP@10 (many valid exact-Lev neighbours).')
retr.to_csv('colab29_retrieval.csv', index=False)

## 10b. All numbers in one place — Spearman + AUROC + MAP@10 + hit@10

Every metric for every method × feed in a single table (and CSV), so the slide numbers don't have to be
read off the bar charts. Reminder on the metric split: **read AA via hit@10** (pair-like), **read SS/3Di
via MAP@10** (neighbourhood). AA has no ≥0.90 pairs, so its 0.90 columns are NaN by construction.

In [ ]:
def _get(df, m, f, col, **filt):
    q = (df.method == m) & (df.feed == f)
    for k, v in filt.items(): q &= (df[k] == v)
    s = df[q][col]
    return float(s.values[0]) if len(s) else np.nan

summary = pd.DataFrame([dict(
    feed=f, method=m,
    spearman=_get(spear, m, f, 'rho'),
    AUROC_rand=_get(auroc, m, f, 'auroc_random'),
    AUROC_hard=_get(auroc, m, f, 'auroc_hard'),
    **{'MAP@0.70': _get(retr, m, f, 'map', bar=0.70), 'MAP@0.70_lo': _get(retr, m, f, 'map_lo', bar=0.70),
       'MAP@0.70_hi': _get(retr, m, f, 'map_hi', bar=0.70), 'hit@0.70': _get(retr, m, f, 'hit', bar=0.70),
       'MAP@0.90': _get(retr, m, f, 'map', bar=0.90), 'hit@0.90': _get(retr, m, f, 'hit', bar=0.90)})
    for f in FEEDS for m in ACTIVE])

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
print('=' * 108)
print('CONSOLIDATED METRICS — Spearman (stratified) | AUROC (full-pool) | retrieval (full-pool)')
print('read AA via hit@10 (pair-like)  |  read SS/3Di via MAP@10 (neighbourhood)  |  AA has no >=0.90 pairs')
print('=' * 108)
print(summary.round(3).to_string(index=False))
summary.round(4).to_csv('colab29_all_metrics.csv', index=False)
print('\nSaved colab29_all_metrics.csv')

## 11. Figures — despined, presentation style

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11})
def despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
COLORS = {'trigram':'#9467bd','Dice':'#8c564b','length':'#888888','ESM2':'#2ca02c',
          'SNN':'#1f77b4','ProtTrans':'#e377c2'}

# --- Centerpiece: Spearman heatmap-style table ---
fig, ax = plt.subplots(figsize=(6.2, 0.7*len(ACTIVE)+1.4))
M = SPEAR_TAB.values.astype(float)
im = ax.imshow(M, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(FEEDS))); ax.set_xticklabels(FEEDS)
ax.set_yticks(range(len(SPEAR_TAB.index))); ax.set_yticklabels(SPEAR_TAB.index)
for r in range(M.shape[0]):
    for c in range(M.shape[1]):
        v = M[r, c]; ax.text(c, r, '—' if np.isnan(v) else f'{v:.2f}', ha='center', va='center',
                             color='white' if (np.isnan(v) or v < 0.6) else 'black', fontsize=11)
ax.set_title('Spearman ρ(sim, normLev) — stratified full-range', fontsize=12)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='ρ')
plt.tight_layout(); plt.savefig('colab29_spearman.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# --- AUROC grouped bars (is the task easy? trigram/Dice baseline) ---
x = np.arange(len(FEEDS)); w = 0.8/len(ACTIVE)
fig, ax = plt.subplots(figsize=(9, 5))
for k, m in enumerate(ACTIVE):
    vals = [float(AUROC_TAB.loc[m, f]) for f in FEEDS]
    ax.bar(x + (k-(len(ACTIVE)-1)/2)*w, vals, w, label=m, color=COLORS.get(m, None))
ax.axhline(0.5, ls='--', color='grey', lw=1); ax.set_ylim(0, 1.08)
ax.set_xticks(x); ax.set_xticklabels(FEEDS); ax.set_ylabel('AUROC (is-high ≥ 0.70)')
ax.set_title('Pairwise discrimination — SNN vs baselines (alphabet³: trigram collapses on SS)')
ax.legend(ncol=len(ACTIVE), fontsize=9, loc='lower center'); despine(ax)
plt.tight_layout(); plt.savefig('colab29_auroc.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# --- MAP@10 grouped bars with CI, per bar ---
def map_chart(bar):
    sub = retr[retr.bar == bar]; x = np.arange(len(FEEDS)); w = 0.8/len(ACTIVE)
    fig, ax = plt.subplots(figsize=(9, 5))
    for k, m in enumerate(ACTIVE):
        p = [float(sub[(sub.method==m)&(sub.feed==f)]['map'].values[0]) for f in FEEDS]
        lo = [float(sub[(sub.method==m)&(sub.feed==f)]['map_lo'].values[0]) for f in FEEDS]
        hi = [float(sub[(sub.method==m)&(sub.feed==f)]['map_hi'].values[0]) for f in FEEDS]
        xoff = x + (k-(len(ACTIVE)-1)/2)*w
        ax.bar(xoff, p, w, label=m, color=COLORS.get(m, None))
        ax.errorbar(xoff, p, yerr=np.vstack([np.array(p)-np.array(lo), np.array(hi)-np.array(p)]),
                    fmt='none', ecolor='black', capsize=2, lw=0.8)
    ax.set_ylim(0, 1.05); ax.set_xticks(x); ax.set_xticklabels(FEEDS)
    ax.set_ylabel('MAP@10 (vs exact-Lev neighbour set)')
    ax.set_title(f'Full-pool retrieval MAP@10 — bar {bar:.2f}' +
                 ('  (operational high-sim)' if bar == 0.70 else '  (well-posed near-identical)'))
    ax.legend(ncol=len(ACTIVE), fontsize=9, loc='upper right'); despine(ax)
    plt.tight_layout(); plt.savefig(f'colab29_map_{int(bar*100)}.png', dpi=150, bbox_inches='tight'); plt.show()
map_chart(0.70); map_chart(0.90)

## 11b. Exhaustive natural-CATH normLev distribution — *why the AA data is hard*

Streams **every unordered pair** of a feed's pool (no full matrix stored) and bins normLev. This proves
the [0.4–0.7] gap in AA is real, not a sampling artefact: CATH-S20 is redundancy-reduced, so natural AA
is a mass of low-similarity pairs, ~5 high-sim pairs, and an **empty middle** — which is exactly why the
full-range training/eval must be synthetic. Default AA-only (~1–2 min); add 'SS'/'3Di' to `PROFILE_FEEDS`
to contrast (they are much fatter).

In [ ]:
PROFILE_FEEDS = ['AA']   # add 'SS','3Di' to profile them too (each is another exhaustive pass)
def exhaustive_normlev_hist(feed, block=512, nbins=20, keep_ex=5):
    seqs = POOL_SEQ[feed]; ids = POOL_IDS[feed]; lens = POOL_LEN[feed]; N = len(seqs)
    edges = np.linspace(0, 1, nbins+1); counts = np.zeros(nbins, dtype=np.int64)
    examples = {b: [] for b in range(nbins)}; total = N*(N-1)//2
    for r0 in range(0, N, block):
        r1 = min(r0+block, N)
        D = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - D/den
        for a in range(r1-r0):
            i = r0+a
            if i+1 >= N: continue
            vals = sim[a, i+1:]; js = np.arange(i+1, N)
            bb = np.clip(np.digitize(vals, edges)-1, 0, nbins-1)
            counts += np.bincount(bb, minlength=nbins)
            for b in range(nbins):
                if len(examples[b]) >= keep_ex: continue
                for h in np.where(bb == b)[0][:keep_ex-len(examples[b])]:
                    j = int(js[h]); examples[b].append(dict(domain_a=ids[i], domain_b=ids[j],
                        normLev=round(float(vals[h]), 4), len_a=int(lens[i]), len_b=int(lens[j])))
    dist = pd.DataFrame([dict(bin=f'[{edges[b]:.2f},{edges[b+1]:.2f})', lo=float(edges[b]),
                              count=int(c), percent=100*c/total) for b, c in enumerate(counts)])
    ex = pd.concat([pd.DataFrame(v) for v in examples.values() if v], ignore_index=True) if any(examples.values()) else pd.DataFrame()
    return dist, ex, total

for feed in PROFILE_FEEDS:
    dist, ex, total = exhaustive_normlev_hist(feed)
    print('='*68); print(f'EXHAUSTIVE {feed} normLev distribution — ALL {total:,} unordered pool pairs'); print('='*68)
    print(dist[['bin', 'count', 'percent']].to_string(index=False))
    print(f'\n  pairs >= 0.70: {int(dist[dist.lo >= 0.70]["count"].sum()):,}  |  '
          f'pairs in [0.40,0.70): {int(dist[(dist.lo >= 0.40) & (dist.lo < 0.70)]["count"].sum()):,} (the empty middle)')
    dist.to_csv(f'colab29_exhaustive_{feed}_normlev_dist.csv', index=False)
    ex.to_csv(f'colab29_exhaustive_{feed}_examples.csv', index=False)
    fig, ax = plt.subplots(figsize=(10, 4.4))
    ax.bar(np.arange(len(dist)), dist['count'].values, color='#1f77b4')
    ax.set_yscale('symlog'); ax.set_xticks(np.arange(len(dist)))
    ax.set_xticklabels([f'{lo:.2f}' for lo in dist['lo']], rotation=45, ha='right')
    ax.set_xlabel('normLev bin (lower edge)'); ax.set_ylabel('pair count (symlog)')
    _mid = ' — the middle is empty' if feed == 'AA' else ''   # 'empty middle' is an AA-specific claim
    ax.set_title(f'Natural CATH {feed}: exhaustive normLev distribution ({total:,} pairs){_mid}')
    despine(ax); plt.tight_layout()
    plt.savefig(f'colab29_exhaustive_{feed}_dist.png', dpi=150, bbox_inches='tight'); plt.show()
print('\nData-difficulty slide: natural AA cannot support a full-range correlation analysis ->'
      ' synthetic perturbation is REQUIRED to study the whole edit-distance function.')

## 12. How to read this notebook

- **Centerpiece = §8 Spearman table** (`colab29_spearman.csv/png`). Primary metric, threshold-free.
  Rows = methods, cols = AA/SS/3Di. SNN row answers Q1; the three columns answer Q2 (cross-alphabet,
  secondary tier); SNN vs trigram/Dice/ESM2 answers Q3.
- **AA Spearman is NOT a comparable cell — present AA as a "control."** AA has ~5 natural high-sim pairs
  and an empty [0.4,0.7] middle (see §11b), so the AA column mostly measures low-band ordering, which the
  3-band SNN doesn't preserve (hence SNN AA ρ≈0.07 despite AA AUROC 0.97 + strong AA retrieval). On the
  slide grey the AA column as "range-truncated; see retrieval + synthetic", and let **SS/3Di carry the
  Spearman story** (SNN best). The AA full-range home-alphabet check lives in the colab30 **synthetic**
  ladder, not here.
- **Retrieval metric split (§10):** report **AA as hit@10** (pair-like, one true partner), **SS/3Di as
  MAP@10** (many valid neighbours; hit@10 is too forgiving there). Set-based win is real: state the metric
  ("set-based exact-Levenshtein oracle") every time — SNN > ESM2 on SS/3Di, dominant at bar 0.90.
- **§11b is the "why the data is hard" slide** (`colab29_exhaustive_AA_*`): the exhaustive natural-AA
  distribution — motivates the synthetic training set.
- **§9 AUROC is full-pool exhaustive** (the whole pairwise grid, not the Spearman sample) — the
  make-or-break "is the task easy?" check. Watch the **alphabet³** effect: the trigram baselines should
  be strong on AA/3Di and collapse toward chance on SS (3³=27 trigrams). If the SNN holds up on SS
  where the k-mer baselines fail, that's the differentiated result. Headline bars = high vs random
  negative; the printed hard-negative [0.30,0.70) table is the harder, more honest contrast.
  Note `trigram` is the raw shared-3-gram COUNT (length-biased, as the professor asked); `Dice` is the
  length-fair trigram comparator — compare the two to separate "k-mer info" from "length artefact."
- **§10 retrieval** reproduces the de-hubbed colab24f numbers (bars 0.70 + 0.90) with the length
  floor as a method row; every method is ranked over the *same* full pool and oracle.
- **Ground truth is normLev only.** Fenoy's 0.66 (vs BLAST) is motivation, never compared directly.
- **To add ProtTrans:** set `ENABLE_PROTTRANS=True` (§1). To add any other method: append one entry
  to `METHODS` with a `pair` and `query` function; all tables + figures pick it up automatically.
- **Caveat (AA top bins):** AA has only ~6 natural high-sim pairs, so its high normLev bins in §6 are
  sparse even after folding in the oracle positives — the printed per-bin counts make this explicit.
  This is the data reality that motivates the synthetic *training* set; the *evaluation* stays on real
  strings.